In [1]:
%pip install -q ezkl onnx onnxruntime psutil numpy pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, json, time, inspect, threading
from pathlib import Path

import numpy as np
import pandas as pd
import onnxruntime as ort
import psutil
import ezkl

MODEL   = "nlinear"
LOGROWS = 12
SCALE   = 13

model_dir = Path("../../models_fused") / MODEL
all_sites = sorted(model_dir.glob("site_*"))
assert all_sites, f"no trained sites under {model_dir}, run prep/train_export.py first"

# a short list for a quick test, or every site for the full run
# sites = all_sites[:3]
sites = all_sites

work_dir    = Path("_work"); work_dir.mkdir(exist_ok=True)
results_dir = Path("../../results"); results_dir.mkdir(parents=True, exist_ok=True)

def site_id(path): return path.name.replace("site_", "")

# ezkl mixes sync and async across versions, so await whatever is awaitable
async def call(fn, *args, **kwargs):
    out = fn(*args, **kwargs)
    return await out if inspect.isawaitable(out) else out

print(len(all_sites), "sites available |", len(sites), "selected")

321 sites available | 321 selected


In [3]:
settings_path = str(work_dir / "settings.json")
srs_path      = str(work_dir / "kzg.srs")
sample_site   = sites[0]

run_args = ezkl.PyRunArgs()
run_args.input_visibility  = "private"
run_args.param_visibility  = "fixed"
run_args.output_visibility = "public"
run_args.input_scale = SCALE
run_args.param_scale = SCALE

assert await call(ezkl.gen_settings, str(sample_site / "network.onnx"), settings_path, py_run_args=run_args)
await call(ezkl.calibrate_settings, str(sample_site / "input.json"), str(sample_site / "network.onnx"), settings_path, "resources")

# pin logrows so a single locally generated SRS serves every site
settings = json.load(open(settings_path)); settings["run_args"]["logrows"] = LOGROWS
json.dump(settings, open(settings_path, "w"))
ezkl.gen_srs(srs_path, LOGROWS)

print("settings ready | logrows:", settings["run_args"]["logrows"],
      "| input_scale:", settings["run_args"]["input_scale"])



 <------------- Numerical Fidelity Report (input_scale: 13, param_scale: 13, scale_input_multiplier: 10) ------------->

+-----------------+-----------------+--------------+----------------+----------------+------------------+---------------+----------------+--------------------+--------------------+------------------------+
| mean_error      | median_error    | max_error    | min_error      | mean_abs_error | median_abs_error | max_abs_error | min_abs_error  | mean_squared_error | mean_percent_error | mean_abs_percent_error |
+-----------------+-----------------+--------------+----------------+----------------+------------------+---------------+----------------+--------------------+--------------------+------------------------+
| 0.0000064295405 | -0.000023134053 | 0.0003914833 | -0.00031019747 | 0.0001544198   | 0.000023134053   | 0.0003914833  | 0.000011593103 | 0.00000003220932   | -0.00016154375     | 0.0011161638           |
+-----------------+-----------------+--------------+-

settings ready | logrows: 12 | input_scale: 13


In [4]:
process = psutil.Process()

# run fn while sampling peak memory, returning wall time, peak MB, and CPU percent
def profile(fn):
    peak = [process.memory_info().rss]
    stop = threading.Event()
    def sample():
        while not stop.is_set():
            peak[0] = max(peak[0], process.memory_info().rss)
    sampler = threading.Thread(target=sample, daemon=True); sampler.start()
    cpu_start, wall_start = process.cpu_times(), time.perf_counter()
    try:
        out = fn()
    finally:
        stop.set(); sampler.join()
    wall = time.perf_counter() - wall_start
    cpu_end = process.cpu_times()
    cpu_time = (cpu_end.user - cpu_start.user) + (cpu_end.system - cpu_start.system)
    return out, wall, peak[0] / 1024**2, (100 * cpu_time / wall if wall else None)

compiled_path = str(work_dir / "network.compiled")
vk_path, pk_path = str(work_dir / "vk.key"), str(work_dir / "pk.key")
witness_path = str(work_dir / "witness.json")
proof_path   = str(work_dir / "proof.json")

rows = []
for site_dir in sites:
    onnx_path, input_path = str(site_dir / "network.onnx"), str(site_dir / "input.json")
    meta = json.load(open(site_dir / "meta.json"))

    assert ezkl.compile_circuit(onnx_path, compiled_path, settings_path)
    assert ezkl.setup(compiled_path, vk_path, pk_path, srs_path)
    await call(ezkl.gen_witness, input_path, compiled_path, witness_path)

    _, prove_s, peak_mb, cpu_pct = profile(
        lambda: ezkl.prove(witness_path, compiled_path, pk_path, proof_path, srs_path=srs_path))

    verify_start = time.perf_counter()
    verified = ezkl.verify(proof_path, settings_path, vk_path, srs_path)
    verify_s = time.perf_counter() - verify_start

    # accuracy gap between the float forecast and the quantized one in the proof
    mean_err = median_err = None
    try:
        session = ort.InferenceSession(onnx_path)
        window = np.array(json.load(open(input_path))["input_data"][0], np.float32).reshape([1, meta["seq_len"], 1])
        float_out = np.array(session.run(None, {session.get_inputs()[0].name: window})[0]).ravel()
        quant_out = np.array(json.load(open(proof_path))["pretty_public_inputs"]["rescaled_outputs"][0], float).ravel()
        n = min(float_out.size, quant_out.size); diff = np.abs(float_out[:n] - quant_out[:n])
        mean_err, median_err = float(diff.mean()), float(np.median(diff))
    except Exception as err:
        print(f"site {site_id(site_dir)} accuracy skipped:", err)

    rows.append({
        "framework": "ezkl", "model": MODEL, "site": meta["site"],
        "params": meta["params"], "logrows": LOGROWS,
        "prove_s": round(prove_s, 4), "verify_s": round(verify_s, 4),
        "cpu_pct": round(cpu_pct, 1) if cpu_pct else None,
        "peak_mem_mb": round(peak_mb, 1), "proof_kb": round(os.path.getsize(proof_path) / 1024, 3),
        "mean_abs_err": mean_err, "median_abs_err": median_err,
        "mse_float": meta["mse_float"], "verified": bool(verified),
    })
    print(f"site {site_id(site_dir):>3}: prove={prove_s:.3f}s verify={verify_s:.3f}s "
          f"mem={peak_mb:.0f}MB cpu={cpu_pct:.0f}% proof={rows[-1]['proof_kb']}KB verified={verified}")

Using 5 columns for range-check.


Using 5 columns for range-check.


Using 5 columns for range-check.


Using 5 columns for range-check.


Using 5 columns for range-check.


...

Using 5 columns for range-check.


Using 5 columns for range-check.


Using 5 columns for range-check.


Using 5 columns for range-check.


site OT accuracy skipped: [ONNXRuntimeError] : 2 : INVALID_ARGUMENT : Invalid rank for input: input Got: 3 Expected: 2 Please fix either the inputs/outputs or the model.
site  OT: prove=0.829s verify=0.032s mem=346MB cpu=483% proof=29.846KB verified=True


## 4. Save and summarize

One CSV per model, one row per site. The analysis notebook reads every
`results/<framework>_<model>.csv` to build the cross-framework plots.

In [5]:
results = pd.DataFrame(rows)
out_path = results_dir / f"ezkl_fused_{MODEL}.csv"
results.to_csv(out_path, index=False)
print("wrote", out_path, "(", len(results), "rows )")
results[["prove_s", "verify_s", "cpu_pct", "peak_mem_mb", "proof_kb",
         "mean_abs_err", "median_abs_err"]].describe().round(3)

wrote ../../results/ezkl_fused_nlinear.csv ( 321 rows )


,prove_s,verify_s,cpu_pct,peak_mem_mb,proof_kb
count,321.000,321.000,321.000,321.000,321.000
mean,0.859,0.031,477.228,332.116,29.806
std,0.078,0.005,12.870,12.441,0.041
min,0.757,0.023,420.100,258.500,29.687
25%,0.816,0.029,470.900,325.900,29.781
50%,0.838,0.030,478.000,334.700,29.807
75%,0.870,0.032,483.200,340.700,29.833
max,1.362,0.108,537.400,360.100,29.924
